# Promo Finder: walkthrough
**Problem 1 · find and rank 20–60s microdrama promos that open on a hook and end on a cliffhanger.**

The AI proposes · code checks · a score ranks · a human approves.

| Stage | Who | Cell |
|---|---|---|
| 1 Parse screenplays → events with IDs | code | §2 |
| 2 Propose promos as event-ID ranges | **LLM** (one call per episode) | §3 |
| 3 Validate: snap, hard rejects, soft flags, dedupe | code | §4 |
| 4 Rank: priority score + diversity | code | §5 |
| 5 Full show run + metrics | code + LLM | §6 |
| 6 Visual prototype of one promo (optional) | LTX-2.3 video model | §7 |

**Self-contained:** §0 writes the whole `promo_finder` package (one cell per module), so nothing is cloned or installed except `gradio_client` for the optional video cell. Runs on Colab, Jupyter or a Hugging Face notebook.

Keys are never stored in this notebook. `GROQ_API_KEY` is read from Colab Secrets, an environment variable, or a hidden prompt; leave it empty to run the rules-only baseline (no AI). `HF_TOKEN` (optional) gives the video preview your own free GPU quota.

## 0 · The code
The complete package. Each cell writes one module to `promo_finder/`.

In [ ]:
import os
os.makedirs("promo_finder", exist_ok=True)
print("writing package to", os.path.abspath("promo_finder"))

In [ ]:
%%writefile promo_finder/__init__.py
# package marker

**`parse.py`** · Stage 1 · parse screenplays into timed events with IDs (code)

In [ ]:
%%writefile promo_finder/parse.py
"""Deterministic screenplay parser. No AI here: the format is regular, so code is cheaper and exact."""
from __future__ import annotations

import re
from dataclasses import dataclass, field, asdict
from pathlib import Path

TS = r"(\d{2}):(\d{2}\.\d{2})"
BLOCK_RE = re.compile(r"^\*\*(?P<head>[^*]+?):?\*\*\s*$")
SE_RE = re.compile(rf"^Start:\s*\({TS}\)\s*\|\s*End:\s*\({TS}\)")
SCENE_RE = re.compile(r"^SCENE\s+(\d+)")


def to_sec(m: str, s: str) -> float:
    return int(m) * 60 + float(s)


def fmt(t: float) -> str:
    m, s = divmod(max(t, 0.0), 60)
    return f"{int(m):02d}:{s:05.2f}"


@dataclass
class Event:
    kind: str            # DIALOGUE | ACTION | CAMERA | SOUND
    start: float
    end: float
    text: str
    speaker: str = ""    # dialogue only
    tone: str = ""       # e.g. "Cold, Commanding"
    scene: int = 0
    id: str = ""         # E001.. over non-camera events, in time order (what the LLM refers to)


@dataclass
class Episode:
    number: int
    file: str
    logline: str
    continuity: str
    duration: float
    events: list[Event] = field(default_factory=list)

    def window(self, start: float, end: float, kinds=("DIALOGUE", "ACTION", "SOUND")) -> list[Event]:
        return [e for e in self.events if e.kind in kinds and e.end > start and e.start < end]

    def to_dict(self):
        return asdict(self)


def _head(head: str) -> tuple[str, str, str]:
    """Return (kind, speaker, tone) for a block header like 'LADY JULIANA (Confused, Breathless)'."""
    h = head.strip().rstrip(":")
    if h == "CAMERA":
        return "CAMERA", "", ""
    if h == "ACTION":
        return "ACTION", "", ""
    if h.startswith("SOUND"):
        return "SOUND", "", ""
    m = re.match(r"^(.*?)\s*\((.*)\)\s*$", h)
    if m:
        return "DIALOGUE", m.group(1).strip(), m.group(2).strip()
    return "DIALOGUE", h, ""


def parse_episode(path: str | Path) -> Episode:
    path = Path(path)
    lines = path.read_text(encoding="utf-8").splitlines()
    meta = {"EPISODE NUMBER": "", "EPISODE LOGLINE": "", "CONTINUITY NOTES": ""}
    for ln in lines[:10]:
        for k in meta:
            if ln.startswith(k + ":"):
                meta[k] = ln.split(":", 1)[1].strip()
    num_m = re.search(r"(\d+)", meta["EPISODE NUMBER"]) or re.search(r"episode-(\d+)", path.name)
    ep = Episode(number=int(num_m.group(1)) if num_m else 0, file=path.name,
                 logline=meta["EPISODE LOGLINE"], continuity=meta["CONTINUITY NOTES"], duration=0.0)

    scene, i, n = 0, 0, len(lines)
    while i < n:
        ln = lines[i].strip()
        if (sm := SCENE_RE.match(ln)):
            scene = int(sm.group(1))
        elif ln.startswith("Scene end:"):
            mm = re.search(TS, ln)
            if mm:
                ep.duration = max(ep.duration, to_sec(*mm.groups()))
        elif (bm := BLOCK_RE.match(ln)) and i + 1 < n and (tm := SE_RE.match(lines[i + 1].strip())):
            kind, speaker, tone = _head(bm.group("head"))
            g = tm.groups()
            body, j = [], i + 2
            while j < n and lines[j].strip() and not BLOCK_RE.match(lines[j].strip()):
                body.append(lines[j].strip())
                j += 1
            text = " ".join(body).strip().strip('"')
            ep.events.append(Event(kind, to_sec(g[0], g[1]), to_sec(g[2], g[3]), text, speaker, tone, scene))
            i = j
            continue
        i += 1
    ep.events.sort(key=lambda e: (e.start, e.end))
    k = 0
    for e in ep.events:
        if e.kind != "CAMERA":
            k += 1
            e.id = f"E{k:03d}"
    if not ep.duration and ep.events:
        ep.duration = max(e.end for e in ep.events)
    return ep


def load_show(folder: str | Path) -> list[Episode]:
    eps = [parse_episode(p) for p in sorted(Path(folder).glob("*.txt"))]
    return sorted(eps, key=lambda e: e.number)


def compact_transcript(ep: Episode, with_camera: bool = False, seconds: bool = False) -> str:
    """Token-cheap view of an episode for the LLM: one timestamped line per event."""
    kinds = {"DIALOGUE", "ACTION", "SOUND"} | ({"CAMERA"} if with_camera else set())
    out = []
    for e in ep.events:
        if e.kind not in kinds:
            continue
        stamp = f"{e.id} [{e.start:.1f}s-{e.end:.1f}s]" if seconds else f"{e.id} [{fmt(e.start)}-{fmt(e.end)}]"
        if e.kind == "DIALOGUE":
            tone = f" ({e.tone})" if e.tone else ""
            out.append(f'{stamp} {e.speaker}{tone}: "{e.text}"')
        else:
            out.append(f"{stamp} {e.kind}: {e.text}")
    return "\n".join(out)


HONORIFICS = {"LADY", "LORD", "SIR", "THE", "MR", "MRS", "MISS", "DAME", "DUKE", "DUCHESS", "YOUNG", "OLD"}


def normalise_speaker(name: str) -> str:
    """'LADY JULIANA' -> 'JULIANA'. Role names like 'THE FORMER GROOM' stay unresolved (known limitation)."""
    words = [w for w in re.split(r"\s+", name.upper().strip()) if w]
    core = [w for w in words if w not in HONORIFICS]
    return " ".join(core or words)


def parse_report(eps: list[Episode]) -> dict:
    """Sanity report; callers should stop if any episode has zero events (format changed)."""
    rows = []
    for e in eps:
        kinds = {}
        for ev in e.events:
            kinds[ev.kind] = kinds.get(ev.kind, 0) + 1
        rows.append({"episode": e.number, "file": e.file, "duration": round(e.duration, 2),
                     "events": len(e.events), **kinds})
    empty = [r["file"] for r in rows if r["events"] == 0]
    return {"episodes": len(eps), "events": sum(r["events"] for r in rows),
            "minutes": round(sum(e.duration for e in eps) / 60, 1),
            "empty_files": empty, "per_episode": rows}

**`llm.py`** · One OpenAI-compatible client: Groq / OpenRouter / OpenAI / Ollama / custom

In [ ]:
%%writefile promo_finder/llm.py
"""One OpenAI-compatible client for every provider (Groq, OpenRouter, OpenAI, Ollama, any custom endpoint).

Swapping models = changing env vars / CLI flags. No provider-specific code paths.
  LLM_PROVIDER  groq | openrouter | openai | ollama | custom | none
  LLM_BASE_URL  overrides the preset (required for custom)
  LLM_API_KEY   or the provider-specific var (GROQ_API_KEY, OPENROUTER_API_KEY, OPENAI_API_KEY)
  LLM_MODEL     overrides the preset model
"""
from __future__ import annotations

import json
import os
import re
import time
from dataclasses import dataclass, field

import requests

PRESETS = {
    "groq": {"base_url": "https://api.groq.com/openai/v1", "model": "openai/gpt-oss-120b", "key_env": "GROQ_API_KEY"},
    "openrouter": {"base_url": "https://openrouter.ai/api/v1", "model": "openai/gpt-oss-120b:free", "key_env": "OPENROUTER_API_KEY"},
    "openai": {"base_url": "https://api.openai.com/v1", "model": "gpt-4o-mini", "key_env": "OPENAI_API_KEY"},
    "ollama": {"base_url": "http://localhost:11434/v1", "model": "llama3.1:8b", "key_env": ""},
    "custom": {"base_url": "", "model": "", "key_env": ""},
    "none": {"base_url": "", "model": "rules-only-baseline", "key_env": ""},
}


class LLMError(RuntimeError):
    pass


class DailyLimitError(LLMError):
    """Provider's per-day quota is used up: stop cleanly; the cache lets the run resume later."""


@dataclass
class LLMConfig:
    provider: str = "groq"
    base_url: str = ""
    model: str = ""
    api_key: str = ""
    backup_keys: list = field(default_factory=list)   # failover ONLY for invalid/revoked keys (401/403)
    temperature: float = 0.0
    timeout: int = 120
    max_retries: int = 5
    usage: dict = field(default_factory=lambda: {"calls": 0, "prompt_tokens": 0, "completion_tokens": 0, "retries": 0})

    @property
    def enabled(self) -> bool:
        return self.provider != "none"

    @classmethod
    def from_env(cls, provider: str | None = None, model: str | None = None,
                 base_url: str | None = None, api_key: str | None = None) -> "LLMConfig":
        provider = (provider or os.getenv("LLM_PROVIDER") or "groq").lower()
        if provider not in PRESETS:
            raise LLMError(f"Unknown provider '{provider}'. Choose from {list(PRESETS)}.")
        p = PRESETS[provider]
        key = api_key or os.getenv("LLM_API_KEY") or (os.getenv(p["key_env"]) if p["key_env"] else "") or ""
        keys = [k.strip() for k in key.split(",") if k.strip()]
        key = keys[0] if keys else ""
        cfg = cls(provider=provider, backup_keys=keys[1:],
                  base_url=(base_url or os.getenv("LLM_BASE_URL") or p["base_url"]).rstrip("/"),
                  model=model or os.getenv("LLM_MODEL") or p["model"],
                  api_key=key)
        if cfg.enabled and not cfg.base_url:
            raise LLMError("custom provider needs LLM_BASE_URL (or --base-url).")
        if cfg.enabled and not cfg.model:
            raise LLMError("No model set: pass --model or LLM_MODEL.")
        if cfg.enabled and provider not in ("ollama", "custom") and not cfg.api_key:
            raise LLMError(f"No API key: set {p['key_env'] or 'LLM_API_KEY'} (or use --provider none).")
        return cfg

    def describe(self) -> dict:
        return {"provider": self.provider, "base_url": self.base_url, "model": self.model, "temperature": self.temperature}


def extract_json(text: str) -> dict:
    """Parse JSON even if the model wrapped it in prose or ```json fences."""
    text = re.sub(r"<think>.*?(</think>|$)", "", text, flags=re.S).strip()   # reasoning models (e.g. Qwen) may think aloud
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"[\{\[].*[\}\]]", text, re.S)
        if not m:
            raise
        return json.loads(m.group(0))


def _post(cfg: LLMConfig, payload: dict) -> requests.Response:
    headers = {"Content-Type": "application/json"}
    if cfg.api_key:
        headers["Authorization"] = f"Bearer {cfg.api_key}"
    return requests.post(f"{cfg.base_url}/chat/completions", headers=headers, json=payload, timeout=cfg.timeout)


def chat_json(cfg: LLMConfig, system: str, user: str, max_tokens: int = 2500) -> tuple[dict, str]:
    """Return (parsed_json, raw_text). Handles 429/5xx backoff, endpoints without JSON mode, and one repair retry."""
    if not cfg.enabled:
        raise LLMError("LLM disabled (provider=none).")
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    use_json_mode = True
    raw = ""
    for repair in range(2):                     # attempt 0 = normal, 1 = ask model to fix its JSON
        attempt = 0
        while True:
            payload = {"model": cfg.model, "messages": messages, "temperature": cfg.temperature, "max_tokens": max_tokens}
            if cfg.provider == "groq" and "gpt-oss" in cfg.model:
                payload["reasoning_effort"] = "low"      # reasoning model: keep hidden thinking tokens (and cost) small
            if use_json_mode:
                payload["response_format"] = {"type": "json_object"}
            try:
                r = _post(cfg, payload)
            except requests.RequestException as ex:
                r, err = None, str(ex)
            if r is not None and r.status_code == 200:
                break
            if r is not None and r.status_code == 400 and use_json_mode and re.search(r"response_format|json_validate_failed|failed to generate json", r.text, re.I):
                use_json_mode = False           # endpoint doesn't support JSON mode -> plain prompt + extraction
                continue
            if r is not None and r.status_code in (401, 403) and cfg.backup_keys:
                cfg.api_key = cfg.backup_keys.pop(0)      # bad/revoked key -> fail over to the next one
                continue
            if r is not None and r.status_code == 429 and re.search(r"per day|\bTPD\b|\bRPD\b", r.text, re.I):
                raise DailyLimitError(f"Daily quota reached for {cfg.model}: {r.text[:200]}")
            retryable = r is None or r.status_code in (408, 409, 429, 500, 502, 503, 504)
            attempt += 1
            if not retryable or attempt > cfg.max_retries:
                detail = err if r is None else f"HTTP {r.status_code}: {r.text[:300]}"
                raise LLMError(detail)
            cfg.usage["retries"] += 1
            wait = 2 ** attempt
            if r is not None:
                try:
                    wait = max(wait, float(r.headers.get("retry-after", 0)))
                except ValueError:
                    pass
            time.sleep(min(wait, 90))
        data = r.json()
        cfg.usage["calls"] += 1
        u = data.get("usage") or {}
        cfg.usage["prompt_tokens"] += int(u.get("prompt_tokens") or 0)
        cfg.usage["completion_tokens"] += int(u.get("completion_tokens") or 0)
        raw = (data.get("choices") or [{}])[0].get("message", {}).get("content") or ""
        try:
            return extract_json(raw), raw
        except (json.JSONDecodeError, ValueError):
            messages = messages + [{"role": "assistant", "content": raw},
                                   {"role": "user", "content": "That was not valid JSON. Reply with ONLY the corrected JSON object."}]
    raise LLMError(f"Model did not return valid JSON. Raw start: {raw[:200]!r}")

**`prompts.py`** · Prompts (versioned)

In [ ]:
%%writefile promo_finder/prompts.py
"""Versioned prompts. Bump PROMPT_VERSION whenever a prompt changes: it is part of the cache key."""

PROMPT_VERSION = "v2"

EPISODE_SYSTEM = """You are a senior promo editor for vertical short-drama ads on TikTok, Instagram Reels and YouTube Shorts.
From ONE episode screenplay, you pick 20-60 second clips that make a scrolling stranger stop and need to download the show.

A good promo:
1. HOOK: its FIRST event grabs attention within 3 seconds - a slap, a shocking line, a reveal, a threat, a public humiliation. Never setup or greetings.
2. CLIFFHANGER: its LAST event is unresolved - a question, threat, interrupted line, shocking reveal or ominous sting - never the answer or the calm after.
3. STANDALONE: a stranger who has never seen the show understands who wants what from the clip itself.

Rules:
- Refer ONLY to event IDs shown (E###). Never write timestamps. Never invent or paraphrase dialogue.
- A promo is usually one continuous range. You MAY join up to 3 ranges from this episode (in story order) if that makes a tighter promo.
- LENGTH IS A HARD RULE: (end time of the last event) - (start time of the first event) must be 20-60 seconds, ideally 30-50.
  Times are shown in seconds, e.g. [69.9s-72.1s]. Compute the length and put it in est_seconds. If it is over 55, start later.
- hook_quote must be copied exactly from the first event; cliffhanger_quote exactly from the last event.
- Return 3 to 5 distinct promos, best first (different moments, not overlapping). Return fewer only if the episode truly lacks strong moments.
- Avoid clips whose main beat is a recap of the previous episode's ending (usually the opening seconds).
- Characters may appear under aliases (e.g. "LADY X" and "X", or a role name like "THE STRANGER").

Score each 1-5 with these anchors:
- hook: 5 = physical shock or explicit reveal in the first line; 3 = tense line; 1 = setup.
- cliffhanger: 5 = question/threat left hanging at the peak; 3 = tense but partly resolved; 1 = resolved or calm.
- standalone: 5 = a stranger understands who wants what; 3 = needs one fact; 1 = needs earlier episodes.

Output JSON only:
{"promos": [{"segments": [{"start_id": "E###", "end_id": "E###"}], "est_seconds": <number>,
  "hook": "<what happens in the first 3 seconds>", "hook_quote": "<exact text of first event>",
  "cliffhanger": "<what is left unresolved>", "cliffhanger_quote": "<exact text of last event>",
  "why": "<ONE sentence on why it works as an ad>",
  "context_needed": "<what a stranger might not understand, or empty>",
  "scores": {"hook": 1-5, "cliffhanger": 1-5, "standalone": 1-5}}]}"""

EPISODE_USER = """SHOW SO FAR (episode loglines, for context only):
{show_context}

EPISODE {number} (duration {duration}). Logline: {logline}
EVENTS:
{transcript}"""

MONTAGE_SYSTEM = """You build montage promos for vertical short-drama ads by JOINING moments from different episodes.
You may ONLY use the validated clips listed. Each clip has an id, an episode, and its events with IDs.

Rules:
- Each montage has 2-3 segments. Each segment is a sub-range of ONE listed clip: {"clip_id", "start_id", "end_id"}, with IDs from that clip.
- Segments must be in story order (earlier episode first).
- The first segment must open on a hook; the last must end unresolved.
- Together the segments must tell ONE situation a stranger can follow (e.g. betrayal -> revenge set-up). If no join is coherent, return none.
- Total 20-60 seconds. Never invent text.
Score hook / cliffhanger / standalone 1-5 (5 best).

Output JSON only:
{"montages": [{"segments": [{"clip_id": "...", "start_id": "E###", "end_id": "E###"}],
  "hook": "...", "cliffhanger": "...", "why": "<ONE sentence>", "context_needed": "...",
  "scores": {"hook": 1-5, "cliffhanger": 1-5, "standalone": 1-5}}]}"""

MONTAGE_USER = """SHOW PREMISE: {premise}

VALIDATED CLIPS:
{clips}

Return 0-3 montages."""

**`candidates.py`** · Stage 2 · LLM proposals + the no-AI baseline

In [ ]:
%%writefile promo_finder/candidates.py
"""Stage 2: propose candidates.

- propose_episode: AI. One call per episode; the model answers with event-ID ranges, never timestamps.
- baseline_episode: no AI. Each episode's own written-in ending (microdrama episodes end on a cliffhanger).
- propose_montages: AI, optional. Joins already-validated clips across episodes.
"""
from __future__ import annotations

import hashlib
import json
from pathlib import Path

from .llm import DailyLimitError, LLMConfig, LLMError, chat_json
from .parse import Episode, compact_transcript, fmt
from .prompts import (EPISODE_SYSTEM, EPISODE_USER, MONTAGE_SYSTEM, MONTAGE_USER, PROMPT_VERSION)


def show_context(eps: list[Episode], number: int, last_k: int = 10) -> str:
    """Episode 1 logline + the last k loglines before this episode. Bounded size for long shows."""
    prior = [e for e in eps if e.number < number]
    keep = ([prior[0]] if prior else []) + [e for e in prior[-last_k:] if prior and e is not prior[0]]
    if not keep:
        return "(this is the first episode)"
    return "\n".join(f"Ep{e.number}: {e.logline}" for e in keep)


def promos_from(parsed, key: str = "promos") -> list:
    """Models sometimes return a list ([{"promos": [...]}] or [promo, ...]) instead of {"promos": [...]}. Accept all."""
    if isinstance(parsed, dict):
        v = parsed.get(key, [])
        return v if isinstance(v, list) else []
    if isinstance(parsed, list):
        out = []
        for item in parsed:
            if isinstance(item, dict) and key in item:
                out += promos_from(item, key)
            elif isinstance(item, dict):
                out.append(item)
        return out
    return []


def _cache_key(*parts: str) -> str:
    return hashlib.sha256("\x1f".join(parts).encode()).hexdigest()[:16]


def _norm_promo(p: dict, episode: int, source: str) -> dict:
    segs = p.get("segments")
    if not segs and p.get("start_id"):                      # tolerate the flat {start_id, end_id} shape
        segs = [{"start_id": p.get("start_id"), "end_id": p.get("end_id")}]
    return {
        "source": source,
        "segments": [{"episode": episode, "start_id": str(s.get("start_id", "")).strip().upper(),
                      "end_id": str(s.get("end_id", "")).strip().upper()} for s in (segs or []) if isinstance(s, dict)],
        "hook": str(p.get("hook", "")), "hook_quote": str(p.get("hook_quote", "")),
        "cliffhanger": str(p.get("cliffhanger", "")), "cliffhanger_quote": str(p.get("cliffhanger_quote", "")),
        "why": str(p.get("why", "")), "context_needed": str(p.get("context_needed", "")),
        "scores": p.get("scores") if isinstance(p.get("scores"), dict) else {},
    }


def propose_episode(ep: Episode, eps: list[Episode], cfg: LLMConfig, cache_dir: Path, raw_dir: Path | None = None) -> tuple[list[dict], dict]:
    """Return (raw candidates, status). Cached per (prompt version, model, episode text, context)."""
    user = EPISODE_USER.format(show_context=show_context(eps, ep.number), number=ep.number,
                               duration=f"{ep.duration:.1f}s", logline=ep.logline or "(none)",
                               transcript=compact_transcript(ep, seconds=True))
    key = _cache_key(PROMPT_VERSION, cfg.base_url, cfg.model, EPISODE_SYSTEM, user)
    cache_dir.mkdir(parents=True, exist_ok=True)
    cpath = cache_dir / f"ep{ep.number:03d}_{key}.json"
    status = {"episode": ep.number, "cached": False, "error": ""}
    if cpath.exists():
        rec = json.loads(cpath.read_text())
        status["cached"] = True
    else:
        try:
            parsed, raw = chat_json(cfg, EPISODE_SYSTEM, user)
        except DailyLimitError as ex:
            status["error"], status["daily_limit"] = str(ex)[:300], True
            return [], status
        except LLMError as ex:
            status["error"] = str(ex)[:300]
            return [], status
        rec = {"episode": ep.number, "model": cfg.model, "prompt_version": PROMPT_VERSION, "raw": raw, "parsed": parsed}
        cpath.write_text(json.dumps(rec, indent=2, ensure_ascii=False))
    if raw_dir:
        raw_dir.mkdir(parents=True, exist_ok=True)
        (raw_dir / f"ep{ep.number:03d}.json").write_text(json.dumps(rec, indent=2, ensure_ascii=False))
    promos = promos_from(rec["parsed"])
    out = [_norm_promo(p, ep.number, "llm") for p in promos if isinstance(p, dict)]
    status["proposed"] = len(out)
    return out, status


def baseline_episode(ep: Episode, target: float = 35.0) -> list[dict]:
    """No-AI baseline: the episode's own ending (written as a cliffhanger), starting on a dialogue line ~25-45s earlier."""
    ev = [e for e in ep.events if e.kind != "CAMERA"]
    if not ev:
        return []
    last = ev[-1]
    end_t = last.end
    starts = [e for e in ev if e.kind == "DIALOGUE" and 25.0 <= end_t - e.start <= 45.0]
    if not starts:
        starts = [e for e in ev if 20.0 <= end_t - e.start <= 60.0]
    if not starts:
        return []
    first = min(starts, key=lambda e: abs((end_t - e.start) - target))
    return [{
        "source": "baseline",
        "segments": [{"episode": ep.number, "start_id": first.id, "end_id": last.id}],
        "hook": "Episode-ending stretch (no AI)", "hook_quote": first.text,
        "cliffhanger": "The episode's own written ending", "cliffhanger_quote": last.text,
        "why": "Microdrama episodes are written to end on a cliffhanger, so the ending is a free promo candidate.",
        "context_needed": "", "scores": {},
    }]


def propose_montages(cands: list[dict], eps: list[Episode], cfg: LLMConfig, cache_dir: Path,
                     raw_dir: Path | None = None, top_n: int = 15) -> tuple[list[dict], dict]:
    """Optional: let the model join validated single-episode clips across episodes. Returns raw montages."""
    by_num = {e.number: e for e in eps}
    pool = [c for c in cands if len({s["episode"] for s in c["segments"]}) == 1][:top_n]
    pool.sort(key=lambda c: (c["segments"][0]["episode"], c["segments"][0]["start"]))   # story order helps the model
    blocks = []
    for c in pool:
        ep = by_num[c["segments"][0]["episode"]]
        lines = []
        for s in c["segments"]:
            for e in ep.events:   # only events fully inside the validated clip, so any pick stays inside it
                if e.kind not in ("DIALOGUE", "ACTION", "SOUND") or e.start < s["start"] - 0.01 or e.end > s["end"] + 0.01:
                    continue
                who = f'{e.speaker}: "{e.text}"' if e.kind == "DIALOGUE" else f"{e.kind}: {e.text}"
                lines.append(f"  {e.id} [{fmt(e.start)}-{fmt(e.end)}] {who}")
        blocks.append(f"CLIP {c['id']} (episode {ep.number}, {c['duration']:.0f}s)\n" + "\n".join(lines))
    premise = eps[0].logline if eps else ""
    user = MONTAGE_USER.format(premise=premise, clips="\n\n".join(blocks))
    key = _cache_key(PROMPT_VERSION, cfg.base_url, cfg.model, MONTAGE_SYSTEM, user)
    cache_dir.mkdir(parents=True, exist_ok=True)
    cpath = cache_dir / f"montage_{key}.json"
    status = {"cached": cpath.exists(), "error": "", "pool": len(pool)}
    if cpath.exists():
        rec = json.loads(cpath.read_text())
    else:
        try:
            parsed, raw = chat_json(cfg, MONTAGE_SYSTEM, user, max_tokens=1500)
        except LLMError as ex:
            status["error"] = str(ex)[:300]
            return [], status
        rec = {"model": cfg.model, "prompt_version": PROMPT_VERSION, "raw": raw, "parsed": parsed}
        cpath.write_text(json.dumps(rec, indent=2, ensure_ascii=False))
    if raw_dir:
        (raw_dir / "montage.json").write_text(json.dumps(rec, indent=2, ensure_ascii=False))
    clip_ep = {c["id"]: c["segments"][0]["episode"] for c in pool}
    out = []
    for m in promos_from(rec["parsed"], "montages"):
        if not isinstance(m, dict):
            continue
        segs = []
        for s in m.get("segments", []) or []:
            cid = str(s.get("clip_id", ""))
            segs.append({"episode": clip_ep.get(cid, -1), "clip_id": cid,
                         "start_id": str(s.get("start_id", "")).upper(), "end_id": str(s.get("end_id", "")).upper()})
        r = _norm_promo({**m, "segments": []}, -1, "montage")
        r["segments"] = segs
        out.append(r)
    status["proposed"] = len(out)
    return out, status

**`validate.py`** · Stage 3 · snap, hard rejects, soft flags, dedupe (code)

In [ ]:
%%writefile promo_finder/validate.py
"""Stage 3: deterministic checks. No AI.

Hard REJECT only what is objectively wrong (bad IDs, wrong order, outside 20-60s).
Soft FLAG what is a judgement call (slow open, resolved ending, unexplained names, recap...).
"""
from __future__ import annotations

import re
from difflib import SequenceMatcher

from .parse import Episode, Event, normalise_speaker

MIN_LEN, MAX_LEN = 20.0, 60.0
MIN_SEG = 4.0
HOOK_WINDOW = 3.0
PRONOUNS = {"he", "she", "they", "it", "that", "this", "him", "her", "them", "those", "these"}
THREAT_WORDS = re.compile(r"\b(will|won't|never|unless|or else|dare|kill|destroy|ruin|swear|regret|pay for|mark my|until|before)\b", re.I)
HOOK_ACTION = re.compile(r"\b(slap|strike|struck|hit|punch|shove|grab|seize|yank|throw|hurl|slam|kneel|collapse|faint|scream|"
                         r"gasp|tear|rip|blood|knife|dagger|sword|gun|pistol|kiss|reveal|unmask|burst|storm|choke|fall|shatter|crash)\w*", re.I)
HOOK_LINE = re.compile(r"[?!]|\b(how dare|liar|traitor|murder|divorce|pregnant|dead|kill|never|marry|betray|who are you|get out)\b", re.I)
TENSION = re.compile(r"\b(freez|stare|glare|trembl|shock|horror|stun|gasp|tense|ominous|menac|threat|danger|dread|"
                     r"suspense|thunder|slam)\w*", re.I)
STING = re.compile(r"\b(sting|swell|crescendo|drone|cliffhanger|peak|ominous|suspense|tension|thunder|boom|cut to black)\w*", re.I)
CALM = re.compile(r"\b(fad(e|ing) out|gentle|warm|peaceful|soft(ly)?|calm|smiles? softly|content)\b", re.I)
GENERIC = {"MAID", "JUDGE", "LORD", "NOBLEMAN", "NOBLEWOMAN", "PRIEST", "GUARD", "OFFICIAL", "INFORMANT", "HOUSEKEEPER",
           "MOTHER", "FATHER", "MATRIARCH", "STRANGER", "MAN", "WOMAN", "GROOM", "BRIDE", "SERVANT", "BUTLER", "DOCTOR"}


def norm_text(s: str) -> str:
    return re.sub(r"[^a-z0-9 ]+", "", s.lower()).strip()


def similarity(a: str, b: str) -> float:
    a, b = norm_text(a), norm_text(b)
    if not a or not b:
        return 0.0
    if min(len(a), len(b)) >= 12 and (a in b or b in a):      # substring only counts for real lines, not "no" in "i know"
        return 1.0 if min(len(a), len(b)) >= 0.6 * max(len(a), len(b)) else 0.9
    return SequenceMatcher(None, a, b).ratio()


def known_names(eps: list[Episode]) -> set[str]:
    """Single-word character names that also appear Title-cased in action/camera text (e.g. 'Juliana')."""
    speakers = {normalise_speaker(e.speaker) for ep in eps for e in ep.events if e.kind == "DIALOGUE"}
    single = {s for s in speakers if " " not in s and "'" not in s and s not in GENERIC and len(s) > 2}
    stage_text = " ".join(e.text for ep in eps for e in ep.events if e.kind in ("ACTION", "CAMERA"))
    return {s for s in single if re.search(rf"\b{s.title()}\b", stage_text)}


def _ids(ep: Episode) -> dict[str, Event]:
    return {e.id: e for e in ep.events if e.id}


def snap(ep: Episode, start: float, end: float) -> tuple[float, float]:
    """Never cut mid-line: pull start back / push end out to the edges of any dialogue line crossing the cut."""
    for e in ep.events:
        if e.kind == "DIALOGUE" and e.start < start < e.end:
            start = e.start
        if e.kind == "DIALOGUE" and e.start < end < e.end:
            end = e.end
    return round(max(0.0, start), 2), round(min(ep.duration or end, end), 2)


def _clip_events(ep: Episode, s: dict, kinds=("DIALOGUE", "ACTION", "SOUND")) -> list[Event]:
    return [e for e in ep.events if e.kind in kinds and e.start >= s["start"] - 0.01 and e.end <= s["end"] + 0.01]


def validate(raw: dict, ep_by_num: dict[int, Episode], names: set[str], clip_pool: dict | None = None) -> dict:
    """Return a candidate dict with status 'valid' or 'rejected' (+ reason). Times always come from the screenplay."""
    c = {k: v for k, v in raw.items()}
    c["flags"], c["status"], c["reject_reason"], c["trimmed"] = [], "valid", "", False

    def reject(reason: str) -> dict:
        c["status"], c["reject_reason"] = "rejected", reason
        return c

    segs = []
    if not raw.get("segments"):
        return reject("no_segments")
    if raw["source"] != "montage" and len(raw["segments"]) > 3:
        return reject("too_many_segments")
    for s in raw["segments"]:
        ep = ep_by_num.get(s["episode"])
        if ep is None:
            return reject("unknown_episode_or_clip")
        ids = _ids(ep)
        a, b = ids.get(s["start_id"]), ids.get(s["end_id"])
        if a is None or b is None:
            return reject("unknown_event_id")
        if a.start > b.start:
            return reject("start_after_end")
        st, en = snap(ep, a.start, max(a.end, b.end))
        seg = {"episode": ep.number, "start_id": a.id, "end_id": b.id, "start": st, "end": en}
        if raw["source"] == "montage":
            clip = (clip_pool or {}).get(s.get("clip_id"))
            if not clip or not any(cs["episode"] == ep.number and cs["start"] - 0.01 <= st and en <= cs["end"] + 0.01
                                   for cs in clip["segments"]):
                return reject("montage_segment_not_from_validated_clip")
            if en - st < MIN_SEG:
                return reject("montage_segment_too_short")
        segs.append(seg)

    # story order + no overlap
    for p, q in zip(segs, segs[1:]):
        if q["episode"] < p["episode"] or (q["episode"] == p["episode"] and q["start"] < p["end"]):
            return reject("segments_out_of_order_or_overlap")
    if raw["source"] == "montage" and len({s["episode"] for s in segs}) < 2:
        return reject("montage_single_episode")

    total = sum(s["end"] - s["start"] for s in segs)
    if total > MAX_LEN:
        # Deterministic repair: keep the model's ending (the cliffhanger) and move the first segment's start
        # forward to the earliest strong line/action that makes the total fit. Prefer a hook-like opener.
        ep0 = ep_by_num[segs[0]["episode"]]
        other = total - (segs[0]["end"] - segs[0]["start"])
        options = [e for e in _clip_events(ep0, segs[0], ("DIALOGUE", "ACTION"))
                   if MIN_LEN <= other + segs[0]["end"] - e.start <= MAX_LEN - 2.0 and e.start > segs[0]["start"]]
        strong = [e for e in options if (e.kind == "DIALOGUE" and HOOK_LINE.search(e.text)) or HOOK_ACTION.search(e.text)]
        pick = (strong or options or [None])[0]
        if pick is not None:
            st, _ = snap(ep0, pick.start, segs[0]["end"])
            segs[0]["start"], segs[0]["start_id"], c["trimmed"] = st, pick.id, True
            total = other + segs[0]["end"] - st
    if not (MIN_LEN <= total <= MAX_LEN):
        c["segments"], c["duration"] = segs, round(total, 2)
        return reject("too_long" if total > MAX_LEN else "too_short")
    c["segments"], c["duration"] = segs, round(total, 2)

    # --- soft flags -------------------------------------------------------------------------
    first_ep, last_ep = ep_by_num[segs[0]["episode"]], ep_by_num[segs[-1]["episode"]]
    first_events = _clip_events(first_ep, segs[0])
    all_events = [e for s in segs for e in _clip_events(ep_by_num[s["episode"]], s)]
    last_events = _clip_events(last_ep, segs[-1])
    dialogue = [e for e in all_events if e.kind == "DIALOGUE"]

    # grounding: what the model quoted vs what the screenplay says at those IDs
    start_ev = _ids(first_ep)[segs[0]["start_id"]]
    end_ev = _ids(last_ep)[segs[-1]["end_id"]]
    c["hook_text"], c["cliffhanger_text"] = start_ev.text, end_ev.text
    c["grounding"] = {"hook": round(similarity(raw.get("hook_quote", ""), start_ev.text), 2),
                      "cliffhanger": round(similarity(raw.get("cliffhanger_quote", ""), end_ev.text), 2)}
    if raw["source"] == "llm" and min(c["grounding"].values()) < 0.85:
        c["flags"].append("quote_mismatch")

    opener = next((e for e in first_events if e.kind in ("DIALOGUE", "ACTION")), None)
    if opener is None or opener.start - segs[0]["start"] > HOOK_WINDOW:
        c["flags"].append("slow_open")
    else:  # something happens in 3s - but is it a hook? (shocking line / physical action), not "Here you go."
        early = [e for e in first_events if e.kind in ("DIALOGUE", "ACTION") and e.start - segs[0]["start"] <= HOOK_WINDOW]
        if not any((e.kind == "DIALOGUE" and HOOK_LINE.search(e.text)) or HOOK_ACTION.search(e.text) for e in early):
            c["flags"].append("weak_open")
    first_line = next((e for e in first_events if e.kind == "DIALOGUE"), None)
    early_text = " ".join(e.text for e in first_events if e.start - segs[0]["start"] <= 5.0)
    if first_line and norm_text(first_line.text).split()[:1] and norm_text(first_line.text).split()[0] in PRONOUNS \
            and not any(re.search(rf"\b{n.title()}\b", early_text) for n in names):
        c["flags"].append("opens_on_pronoun")

    # cliffhanger: the last line/action must be unresolved; a closing sound counts only if it is a sting, not a fade-out
    tail = [e for e in last_events if e.start >= segs[-1]["end"] - 6.0] or last_events[-1:] or [end_ev]
    last_beat = next((e for e in reversed(tail) if e.kind in ("DIALOGUE", "ACTION")), tail[-1])
    t = last_beat.text.strip()
    unresolved = (t.endswith(("?", "...", "…", "—", "-", "!")) or bool(THREAT_WORDS.search(t)) or bool(TENSION.search(t))
                  or any(e.kind == "SOUND" and STING.search(e.text) and not CALM.search(e.text) for e in tail))
    if not unresolved or any(CALM.search(e.text) for e in tail if e.kind in ("SOUND", "ACTION")):
        c["flags"].append("resolved_ending")

    present = {normalise_speaker(e.speaker) for e in dialogue}
    stage = " ".join(e.text for s in segs for e in _clip_events(ep_by_num[s["episode"]], s, ("ACTION", "CAMERA")))
    mentioned = {n for n in names for e in dialogue if re.search(rf"\b{n.title()}\b", e.text)}
    absent = sorted(n for n in mentioned if not any(n in p for p in present) and not re.search(rf"\b{n.title()}\b", stage))
    if absent:
        c["flags"].append("unexplained_names")
    c["absent_names"] = absent

    if len(dialogue) < 2:
        c["flags"].append("low_dialogue")

    prev = ep_by_num.get(segs[0]["episode"] - 1)
    if prev and segs[0]["start"] <= 20.0 and dialogue:
        tail = [e.text for e in prev.events if e.kind == "DIALOGUE" and e.start >= prev.duration - 40.0]
        hits = sum(1 for e in dialogue if any(similarity(e.text, t2) >= 0.85 for t2 in tail))
        if hits / len(dialogue) > 0.6:
            c["flags"].append("recap")

    if c["trimmed"]:
        c["flags"].append("auto_trimmed_start")
    c["rule"] = {"hook_ok": not ({"slow_open", "weak_open", "opens_on_pronoun"} & set(c["flags"])),
                 "cliff_ok": "resolved_ending" not in c["flags"],
                 "standalone_ok": "unexplained_names" not in c["flags"]}
    return c


def _event_keys(c: dict, ep_by_num: dict[int, Episode]) -> set:
    return {(s["episode"], e.id) for s in c["segments"] for e in _clip_events(ep_by_num[s["episode"]], s)}


def _dialogue(c: dict, ep_by_num: dict[int, Episode]) -> list[str]:
    """Substantive lines only (>= 4 words): short exclamations repeat across a show and are not evidence of a recap."""
    return [e.text for s in c["segments"] for e in _clip_events(ep_by_num[s["episode"]], s, ("DIALOGUE",))
            if len(e.text.split()) >= 4]


def dedupe(cands: list[dict], ep_by_num: dict[int, Episode]) -> tuple[list[dict], list[dict]]:
    """cands must be sorted best-first. Drops >50% event overlap, and cross-episode recaps (>70% same lines; keep earlier ep)."""
    kept, dropped = [], []
    for c in cands:
        keys, lines = _event_keys(c, ep_by_num), _dialogue(c, ep_by_num)
        dup = None
        for k in kept:
            kk = _event_keys(k, ep_by_num)
            if keys and kk and len(keys & kk) / min(len(keys), len(kk)) > 0.5:
                dup = ("overlap", k["id"])
                break
            kl = _dialogue(k, ep_by_num)
            if lines and kl and k["segments"][0]["episode"] != c["segments"][0]["episode"]:
                hits = sum(1 for a in lines if any(similarity(a, b) >= 0.85 for b in kl))
                if hits / len(lines) > 0.7:
                    dup = ("recap_duplicate", k["id"])
                    break
        if dup:
            c["dropped_as"], c["duplicate_of"] = dup
            dropped.append(c)
        else:
            kept.append(c)
    return kept, dropped

**`rank.py`** · Stage 4 · priority score, diversity selection, sensitivity (code)

In [ ]:
%%writefile promo_finder/rank.py
"""Stage 4: transparent priority score + diverse selection. No AI.

The weights are product judgement, not science. They are configurable, every component is shown,
and sensitivity() reports how much the top 10 moves when weights shift.
"""
from __future__ import annotations

import statistics

WEIGHTS = {"hook": 0.35, "cliffhanger": 0.30, "standalone": 0.25, "funnel": 0.10}
FLAG_PENALTY, FLAG_PENALTY_CAP = 0.05, 0.15
LLM_SHARE = 0.7          # share of each sub-score from the LLM's anchored 1-5 rating; the rest from rule checks


def _llm(c: dict, key: str) -> float | None:
    try:
        v = float(c.get("scores", {}).get(key))
        return min(max((v - 1) / 4, 0.0), 1.0)
    except (TypeError, ValueError):
        return None


def components(c: dict, n_eps: int) -> dict:
    rule = c.get("rule", {})
    out = {}
    for key, rk in (("hook", "hook_ok"), ("cliffhanger", "cliff_ok"), ("standalone", "standalone_ok")):
        r = 1.0 if rule.get(rk) else 0.0
        l = _llm(c, key)
        out[key] = r if l is None else LLM_SHARE * l + (1 - LLM_SHARE) * r   # baseline has no LLM score -> rules only
    ep = min(s["episode"] for s in c["segments"])
    out["funnel"] = 1.0 if n_eps <= 1 else 1 - (ep - 1) / (n_eps - 1)
    return {k: round(v, 3) for k, v in out.items()}


def score(c: dict, n_eps: int, weights: dict | None = None) -> float:
    w = weights or WEIGHTS
    comp = c["components"] if "components" in c else components(c, n_eps)
    penalty = min(FLAG_PENALTY * len([f for f in c.get("flags", []) if f != "low_dialogue"]), FLAG_PENALTY_CAP)
    return round(sum(w[k] * comp[k] for k in w) - penalty, 4)


def apply_scores(cands: list[dict], n_eps: int, weights: dict | None = None) -> list[dict]:
    for c in cands:
        c["components"] = components(c, n_eps)
        c["priority"] = score(c, n_eps, weights)
    return sorted(cands, key=lambda c: -c["priority"])


def select(cands: list[dict], top_k: int = 10, max_per_ep: int = 2) -> list[dict]:
    """Greedy by priority; at most max_per_ep per episode and no repeated hook event in the top_k. Rest follow in order."""
    top, rest, per_ep, hooks = [], [], {}, set()
    for c in cands:
        ep = c["segments"][0]["episode"]
        hk = (ep, c["segments"][0]["start_id"])
        if len(top) < top_k and per_ep.get(ep, 0) < max_per_ep and hk not in hooks:
            top.append(c)
            per_ep[ep] = per_ep.get(ep, 0) + 1
            hooks.add(hk)
        else:
            rest.append(c)
    ranked = top + rest
    for i, c in enumerate(ranked, 1):
        c["rank"] = i
        c["post_reason"] = explain(c)
    return ranked


def explain(c: dict) -> str:
    comp = c["components"]
    labels = {"hook": "strong opening hook", "cliffhanger": "unresolved ending", "standalone": "easy for a stranger to follow",
              "funnel": "early episode (no spoilers, viewer lands near Ep 1)"}
    best = sorted(("hook", "cliffhanger", "standalone", "funnel"), key=lambda k: -comp[k] * WEIGHTS[k])[:2]
    reason = " + ".join(labels[k] for k in best)
    if c.get("flags"):
        reason += f"; check: {', '.join(c['flags'])}"
    return reason


def sensitivity(cands: list[dict], n_eps: int, top_k: int = 10, delta: float = 0.1) -> dict:
    """For each weight +/- delta (renormalised), how many of the top_k stay in the top_k?"""
    base = [c["id"] for c in select(apply_scores([dict(c) for c in cands], n_eps), top_k)[:top_k]]
    res = {}
    for k in WEIGHTS:
        for sign in (+1, -1):
            w = dict(WEIGHTS)
            w[k] = max(0.0, w[k] + sign * delta)
            tot = sum(w.values())
            w = {kk: v / tot for kk, v in w.items()}
            alt = [c["id"] for c in select(apply_scores([dict(c) for c in cands], n_eps, w), top_k)[:top_k]]
            res[f"{k}{'+' if sign > 0 else '-'}{delta}"] = len(set(base) & set(alt))
    # restore ranking/fields for the real weights
    select(apply_scores(cands, n_eps), top_k)
    return {"top_k": top_k, "overlap_with_base": res, "min_overlap": min(res.values()) if res else None}


def score_spread(cands: list[dict]) -> dict:
    out = {}
    for k in ("hook", "cliffhanger", "standalone"):
        vals = [float(c["scores"][k]) for c in cands if isinstance(c.get("scores"), dict) and str(c["scores"].get(k, "")).replace(".", "", 1).isdigit()]
        out[k] = {"n": len(vals), "mean": round(statistics.mean(vals), 2) if vals else None,
                  "stdev": round(statistics.pstdev(vals), 2) if len(vals) > 1 else None}
    return out

**`evaluate.py`** · Automatic metrics + blinded human review sheet

In [ ]:
%%writefile promo_finder/evaluate.py
"""Evaluation. Automatic metrics come from the run itself; human metrics come from a blinded review sheet.

  python -m promo_finder.evaluate sheet  <run_dir> <pack_dir>   -> review_sheet.csv (+ hidden key)
  python -m promo_finder.evaluate score  <run_dir>              -> human_metrics.json (after you fill the sheet)
"""
from __future__ import annotations

import json
import random
import sys
from collections import Counter
from pathlib import Path

import pandas as pd

from .parse import fmt, load_show


def auto_metrics(out: Path, *, llm_raw, llm_checked, llm_kept, llm_dups, ranked, chosen, statuses,
                 llm_enabled, ep_by_num) -> dict:
    out = Path(out)
    m: dict = {"note": "All numbers computed from this run's files; nothing hand-entered."}
    # Hard invariants on what we ship (always checked, also in no-LLM mode)
    bad = []
    for c in ranked:
        tot = sum(s["end"] - s["start"] for s in c["segments"])
        for s in c["segments"]:
            ep = ep_by_num[s["episode"]]
            if not (0 <= s["start"] < s["end"] <= ep.duration + 0.01):
                bad.append((c["id"], "segment_outside_episode"))
        if not (20 <= tot <= 60):
            bad.append((c["id"], "duration_outside_20_60"))
    m["shipped_invariant_violations"] = bad
    m["shipped_promos"] = len(ranked)
    cov = {s["episode"] for c in ranked for s in c["segments"]}
    m["episode_coverage"] = {"episodes_processed": len(chosen), "with_valid_promo": len(cov),
                             "share": round(len(cov) / len(chosen), 3) if chosen else None,
                             "no_promo": [n for n in chosen if n not in cov]}
    if llm_enabled:
        n = len(llm_raw)
        valid = [c for c in llm_checked if c["status"] == "valid"]
        rej = Counter(c["reject_reason"] for c in llm_checked if c["status"] == "rejected")
        m["llm"] = {
            "episodes_failed": [s["episode"] for s in statuses if s.get("error")],
            "proposed": n,
            "validity_rate": round(len(valid) / n, 3) if n else None,
            "passed_without_trim": round(sum(1 for c in valid if not c.get("trimmed")) / n, 3) if n else None,
            "trimmed": sum(1 for c in valid if c.get("trimmed")),
            "rejected_by_reason": dict(rej),
            "id_validity": round(1 - rej.get("unknown_event_id", 0) / n, 3) if n else None,
            "quote_accuracy": round(sum(1 for c in valid if min(c["grounding"].values()) >= 0.85) / len(valid), 3) if valid else None,
            "duplicate_rate": round(len(llm_dups) / len(valid), 3) if valid else None,
            "duplicates_by_type": dict(Counter(c.get("dropped_as") for c in llm_dups)),
            "flag_counts_all_valid": dict(Counter(f for c in llm_kept for f in c["flags"])),
            "top15_flagged_for_review": sum(1 for c in ranked[:15] if c["flags"]),
            "avg_promos_per_episode": round(n / len(chosen), 2) if chosen else None,
        }
    (out / "metrics.json").write_text(json.dumps(m, indent=2))
    return m


def _excerpt(c: dict, eps_by_num) -> str:
    lines = []
    for s in c["segments"]:
        ep = eps_by_num[s["episode"]]
        lines.append(f"--- Episode {s['episode']} {fmt(s['start'])}-{fmt(s['end'])} ---")
        for e in ep.events:
            if e.kind in ("DIALOGUE", "ACTION", "SOUND") and e.start >= s["start"] - 0.01 and e.end <= s["end"] + 0.01:
                t = f"+{e.start - s['start']:.1f}s"
                lines.append(f"{t} {e.speaker}: \"{e.text}\"" if e.kind == "DIALOGUE" else f"{t} [{e.kind}] {e.text}")
    return "\n".join(lines)


def make_review_sheet(run_dir: str | Path, pack_dir: str | Path, seed: int = 7) -> Path:
    """Blinded, shuffled: LLM top 15 + 5 random LLM ranked >=30 + baseline top 10. Rater sees only excerpts."""
    run_dir = Path(run_dir)
    eps = {e.number: e for e in load_show(pack_dir)}
    llm = json.loads((run_dir / "promos_ranked.json").read_text())
    base = json.loads((run_dir / "baseline_ranked.json").read_text())
    rng = random.Random(seed)
    picks = [("llm_top", c) for c in llm[:15] if c["source"] == "llm"]
    low = [c for c in llm[29:] if c["source"] == "llm"]
    picks += [("llm_low", c) for c in rng.sample(low, min(5, len(low)))]
    picks += [("baseline", c) for c in base[:10]]
    rng.shuffle(picks)
    rows, key = [], []
    for i, (group, c) in enumerate(picks, 1):
        rows.append({"item": i, "excerpt": _excerpt(c, eps), "hook_1to3": "", "cliffhanger_1to3": "",
                     "standalone_1to3": "", "post_it_YN": "", "boundary_ok_YN": "", "notes": ""})
        key.append({"item": i, "group": group, "id": c["id"], "priority": c.get("priority"),
                    "rank": c.get("rank")})
    sheet = run_dir / "review_sheet.csv"
    pd.DataFrame(rows).to_csv(sheet, index=False)
    pd.DataFrame(key).to_csv(run_dir / "review_key_DO_NOT_OPEN_BEFORE_RATING.csv", index=False)
    return sheet


def _spearman(x: list[float], y: list[float]) -> float | None:
    if len(x) < 3:
        return None
    rx = pd.Series(x).rank().tolist()
    ry = pd.Series(y).rank().tolist()
    return round(pd.Series(rx).corr(pd.Series(ry)), 3)


def score_review(run_dir: str | Path) -> dict:
    run_dir = Path(run_dir)
    s = pd.read_csv(run_dir / "review_sheet.csv")
    k = pd.read_csv(run_dir / "review_key_DO_NOT_OPEN_BEFORE_RATING.csv")
    d = s.merge(k, on="item")
    for col in ("hook_1to3", "cliffhanger_1to3", "standalone_1to3"):
        d[col] = pd.to_numeric(d[col], errors="coerce")
    d["total"] = d[["hook_1to3", "cliffhanger_1to3", "standalone_1to3"]].sum(axis=1, min_count=3)
    yes = lambda col: d[col].astype(str).str.strip().str.upper().str.startswith("Y")
    d["post"], d["boundary"] = yes("post_it_YN"), yes("boundary_ok_YN")
    res = {}
    for g, grp in d.groupby("group"):
        res[g] = {"n": int(len(grp)), "mean_hook": round(grp["hook_1to3"].mean(), 2),
                  "mean_cliffhanger": round(grp["cliffhanger_1to3"].mean(), 2),
                  "mean_standalone": round(grp["standalone_1to3"].mean(), 2),
                  "mean_total": round(grp["total"].mean(), 2), "post_rate": round(grp["post"].mean(), 3),
                  "boundary_ok_rate": round(grp["boundary"].mean(), 3)}
    top10 = d[(d["group"] == "llm_top")].sort_values("rank").head(10)
    res["precision_at_10"] = round(top10["post"].mean(), 3) if len(top10) else None
    llm = d[d["group"].isin(["llm_top", "llm_low"])].dropna(subset=["total"])
    res["spearman_priority_vs_human"] = _spearman(llm["priority"].tolist(), llm["total"].tolist())
    (run_dir / "human_metrics.json").write_text(json.dumps(res, indent=2))
    return res


if __name__ == "__main__":
    cmd = sys.argv[1] if len(sys.argv) > 1 else ""
    if cmd == "sheet":
        print(make_review_sheet(sys.argv[2], sys.argv[3]))
    elif cmd == "score":
        print(json.dumps(score_review(sys.argv[2]), indent=2))
    else:
        print(__doc__)

**`pipeline.py`** · Orchestration (shared by CLI, notebook and the Space)

In [ ]:
%%writefile promo_finder/pipeline.py
"""Orchestrates the stages. Shared by run.py (CLI), the notebook and the browser Space (via Pyodide)."""
from __future__ import annotations

import json
import re
import tempfile
import time
import zipfile
from datetime import datetime
from pathlib import Path

import pandas as pd

from . import evaluate
from .candidates import baseline_episode, propose_episode, propose_montages
from .llm import LLMConfig
from .parse import fmt, load_show, parse_report
from .prompts import PROMPT_VERSION
from .rank import apply_scores, score_spread, select, sensitivity
from .validate import dedupe, known_names, validate


def _one_sentence(s: str) -> str:
    s = s.strip()
    m = re.match(r"(.+?[.!?])(\s|$)", s)
    return m.group(1) if m else s


def parse_range(spec: str | None, available: list[int]) -> list[int]:
    if not spec:
        return available
    out = set()
    for part in spec.split(","):
        if "-" in part:
            a, b = part.split("-")
            out |= set(range(int(a), int(b) + 1))
        elif part.strip():
            out.add(int(part))
    return [n for n in available if n in out]


def load_input(path: str | Path):
    """Folder of .txt files, or a .zip of them. Returns (episodes, tempdir-or-None)."""
    path = Path(path)
    tmp = None
    if path.suffix.lower() == ".zip":
        tmp = tempfile.TemporaryDirectory()
        with zipfile.ZipFile(path) as z:
            for name in z.namelist():
                if name.lower().endswith(".txt") and not name.startswith("__MACOSX"):
                    target = Path(tmp.name) / Path(name).name
                    target.write_bytes(z.read(name))
        path = Path(tmp.name)
    eps = load_show(path)
    return eps, tmp


def seg_str(c: dict) -> str:
    return "; ".join(f"Ep{s['episode']} {fmt(s['start'])}-{fmt(s['end'])} [{s['start_id']}-{s['end_id']}]" for s in c["segments"])


def to_rows(cands: list[dict]) -> list[dict]:
    rows = []
    for c in cands:
        comp = c.get("components", {})
        sc = c.get("scores", {}) or {}
        rows.append({
            "rank": c.get("rank"), "id": c["id"], "source": c["source"],
            "episodes": ",".join(str(s["episode"]) for s in c["segments"]),
            "segments": seg_str(c), "duration_s": c.get("duration"),
            "hook": c.get("hook", ""), "hook_line": c.get("hook_text", ""),
            "cliffhanger": c.get("cliffhanger", ""), "cliffhanger_line": c.get("cliffhanger_text", ""),
            "why": c.get("why", ""), "context_needed": c.get("context_needed", ""),
            "priority": c.get("priority"), "post_reason": c.get("post_reason", ""),
            "c_hook": comp.get("hook"), "c_cliffhanger": comp.get("cliffhanger"),
            "c_standalone": comp.get("standalone"), "c_funnel": comp.get("funnel"),
            "llm_hook": sc.get("hook"), "llm_cliffhanger": sc.get("cliffhanger"), "llm_standalone": sc.get("standalone"),
            "flags": ",".join(c.get("flags", [])),
        })
    return rows


def run_pipeline(input_path: str | Path, cfg: LLMConfig, out_root: str | Path = "outputs",
                 episodes: str | None = None, montage: bool = False, cache_dir: str | Path | None = None,
                 price_in: float | None = None, price_out: float | None = None, progress=None,
                 run_name: str | None = None) -> dict:
    t0 = time.time()
    eps, tmp = load_input(input_path)
    try:
        report = parse_report(eps)
        if not eps:
            raise ValueError("No .txt screenplay files found in the input.")
        if report["empty_files"]:
            raise ValueError(f"Parser found 0 events in: {report['empty_files']}. Screenplay format differs from expected.")
        ep_by_num = {e.number: e for e in eps}
        chosen = parse_range(episodes, [e.number for e in eps])
        names = known_names(eps)
        out_root = Path(out_root)
        cache_dir = Path(cache_dir) if cache_dir else out_root / "cache"
        stamp = run_name or f"run_{datetime.now():%Y%m%d_%H%M%S}_{re.sub(r'[^A-Za-z0-9.-]+', '-', cfg.model)}"
        out = out_root / stamp
        raw_dir = out / "raw_llm"
        out.mkdir(parents=True, exist_ok=True)

        llm_raw, statuses, base_raw = [], [], []
        for i, n in enumerate(chosen, 1):
            ep = ep_by_num[n]
            if cfg.enabled and any(x.get("daily_limit") for x in statuses):
                statuses.append({"episode": n, "error": "skipped: daily quota reached - rerun later, cache resumes"})
            elif cfg.enabled:
                props, st = propose_episode(ep, eps, cfg, cache_dir, raw_dir)
                for j, p in enumerate(props, 1):
                    p["id"] = f"L{n:03d}-{j}"
                llm_raw += props
                statuses.append(st)
            for p in baseline_episode(ep):
                p["id"] = f"B{n:03d}"
                base_raw.append(p)
            if progress:
                progress(i, len(chosen), n)

        n_eps = len(eps)
        llm_checked = [validate(p, ep_by_num, names) for p in llm_raw]
        base_checked = [validate(p, ep_by_num, names) for p in base_raw]
        llm_valid = apply_scores([c for c in llm_checked if c["status"] == "valid"], n_eps)
        base_valid = apply_scores([c for c in base_checked if c["status"] == "valid"], n_eps)
        llm_kept, llm_dups = dedupe(llm_valid, ep_by_num)
        base_kept, _ = dedupe(base_valid, ep_by_num)
        llm_ranked = select(llm_kept)
        base_ranked = select(base_kept)
        for c in llm_ranked + base_ranked:
            c["why"] = _one_sentence(c.get("why", ""))
        sens = sensitivity(llm_kept, n_eps) if len(llm_kept) >= 10 else None
        llm_ranked = sorted(llm_kept, key=lambda c: c["rank"])

        mont_ranked, mont_rejected, mont_status = [], [], None
        if montage and cfg.enabled and llm_ranked:
            pool = {c["id"]: c for c in llm_ranked[:15]}
            raw_m, mont_status = propose_montages(llm_ranked, eps, cfg, cache_dir, raw_dir)
            for j, m in enumerate(raw_m, 1):
                m["id"] = f"M{j}"
            checked = [validate(m, ep_by_num, names, clip_pool=pool) for m in raw_m]
            mont_rejected = [c for c in checked if c["status"] == "rejected"]
            mont_ranked = select(apply_scores([c for c in checked if c["status"] == "valid"], n_eps), top_k=10, max_per_ep=99)
            for c in mont_ranked:
                c["why"] = _one_sentence(c.get("why", ""))

        primary = llm_ranked if cfg.enabled else base_ranked
        rejected = [c for c in llm_checked if c["status"] == "rejected"] + llm_dups + mont_rejected

        # ---- write outputs -----------------------------------------------------------------
        pd.DataFrame(to_rows(primary)).to_csv(out / "promos_ranked.csv", index=False)
        (out / "promos_ranked.json").write_text(json.dumps(primary, indent=2, ensure_ascii=False))
        pd.DataFrame(to_rows(base_ranked)).to_csv(out / "baseline_ranked.csv", index=False)
        (out / "baseline_ranked.json").write_text(json.dumps(base_ranked, indent=2, ensure_ascii=False))
        if mont_ranked or mont_rejected:
            pd.DataFrame(to_rows(mont_ranked)).to_csv(out / "montages_ranked.csv", index=False)
            (out / "montages_ranked.json").write_text(json.dumps(mont_ranked, indent=2, ensure_ascii=False))
        (out / "rejected.json").write_text(json.dumps(rejected, indent=2, ensure_ascii=False))
        (out / "parse_report.json").write_text(json.dumps(report, indent=2))

        elapsed = round(time.time() - t0, 1)
        usage = dict(cfg.usage)
        cost = None
        if price_in is not None and price_out is not None:
            cost = round(usage["prompt_tokens"] / 1e6 * price_in + usage["completion_tokens"] / 1e6 * price_out, 4)
        meta = {
            "run": stamp, "created": datetime.now().isoformat(timespec="seconds"), "input": Path(input_path).name,
            "llm": cfg.describe(), "prompt_version": PROMPT_VERSION, "episodes_processed": chosen,
            "usage": usage, "elapsed_s": elapsed, "price_per_1M": {"in": price_in, "out": price_out}, "cost_usd": cost,
            "episode_status": statuses, "montage_status": mont_status,
            "llm_score_spread": score_spread(llm_valid), "weight_sensitivity": sens,
        }
        (out / "run_meta.json").write_text(json.dumps(meta, indent=2))
        metrics = evaluate.auto_metrics(out, llm_raw=llm_raw, llm_checked=llm_checked, llm_kept=llm_kept,
                                        llm_dups=llm_dups, ranked=primary, chosen=chosen, statuses=statuses,
                                        llm_enabled=cfg.enabled, ep_by_num=ep_by_num)
        return {"out_dir": str(out), "ranked": primary, "baseline": base_ranked, "montages": mont_ranked,
                "rejected": rejected, "meta": meta, "metrics": metrics, "report": report, "episodes": eps}
    finally:
        if tmp is not None:
            tmp.cleanup()

**`preview.py`** · Optional: shot prompts for the LTX video preview

In [ ]:
%%writefile promo_finder/preview.py
"""Optional: turn a ranked promo into two short text-to-video shot prompts (hook shot + cliffhanger shot).

Used only for a *visual prototype* of the promo (LTX-2.3 makes ~5-10s clips with synced audio).
Ranking and labels never depend on this. If the LLM is off or fails, a template built from the real
screenplay lines is used instead, so this function always returns prompts.
"""
from __future__ import annotations

from .llm import LLMConfig, LLMError, chat_json
from .parse import Episode

SHOT_SYSTEM = """You write prompts for a text-to-video model that generates ~6 second vertical (9:16) clips WITH synced audio.
You get one promo moment from a drama screenplay. Write TWO shot prompts:
 hook_shot  - the promo's opening moment (the first lines below)
 cliff_shot - the promo's final, unresolved moment (the last lines below)
Each prompt: 45-80 words, present tense, one continuous shot. Describe the setting, the characters' look and positions,
the camera (close-up / push-in), lighting and mood, then the action, then the exact spoken line in double quotes with
who says it and how, plus any sound effect. Use ONLY lines given below; do not invent dialogue. No on-screen text, no logos.
Return JSON only: {"hook_shot": "...", "cliff_shot": "..."}"""


def _lines(ep: Episode, start: float, end: float, limit: int) -> list[str]:
    out = []
    for e in ep.window(start, end):
        if e.kind == "DIALOGUE":
            tone = f" ({e.tone})" if e.tone else ""
            out.append(f'{e.speaker}{tone}: "{e.text}"')
        else:
            out.append(f"[{e.kind.lower()}] {e.text}")
    return out[:limit] if limit > 0 else out[limit:]


def moments(c: dict, ep_by_num: dict[int, Episode]) -> tuple[list[str], list[str], str]:
    """(first ~10s of lines, last ~10s of lines, logline) for a validated candidate."""
    first, last = c["segments"][0], c["segments"][-1]
    ep1, ep2 = ep_by_num[first["episode"]], ep_by_num[last["episode"]]
    head = _lines(ep1, first["start"], min(first["end"], first["start"] + 10), 6)
    tail = _lines(ep2, max(last["start"], last["end"] - 10), last["end"], -6)
    return head, tail, ep1.logline


def _template(head: list[str], tail: list[str], logline: str) -> dict:
    style = ("Vertical 9:16 cinematic drama shot, shallow depth of field, dramatic lighting, "
             "tight close-up with a slow push-in. ")
    ctx = f"Story: {logline.rstrip('. ')}. " if logline else ""
    return {"hook_shot": style + ctx + " ".join(head), "cliff_shot": style + ctx + " ".join(tail) + " Hold on the face; tense silence.",
            "source": "template"}


def shot_prompts(c: dict, ep_by_num: dict[int, Episode], cfg: LLMConfig | None = None) -> dict:
    head, tail, logline = moments(c, ep_by_num)
    if cfg is None or not cfg.enabled:
        return _template(head, tail, logline)
    user = (f"SHOW LOGLINE: {logline}\nHOOK (opening lines):\n" + "\n".join(head) +
            "\n\nCLIFFHANGER (closing lines):\n" + "\n".join(tail))
    try:
        parsed, _ = chat_json(cfg, SHOT_SYSTEM, user, max_tokens=900)
        if isinstance(parsed, dict) and parsed.get("hook_shot") and parsed.get("cliff_shot"):
            return {"hook_shot": str(parsed["hook_shot"]), "cliff_shot": str(parsed["cliff_shot"]), "source": "llm"}
    except LLMError:
        pass
    return _template(head, tail, logline)

In [ ]:
import os, sys, importlib, getpass
sys.path.insert(0, os.getcwd())
importlib.invalidate_caches()
for m in [k for k in sys.modules if k.startswith("promo_finder")]: del sys.modules[m]

def secret(name, ask=False):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception:
        pass
    v = os.environ.get(name)
    if not v and ask:
        v = getpass.getpass(f"{name} (Enter to skip): ").strip() or None
    return v
import promo_finder.pipeline
print("promo_finder ready")

## 1 · Load a show
On Colab you get an upload button (one `.zip` of episode `.txt` files, or several `.txt`). Elsewhere, set `SHOW_SOURCE` to a folder or `.zip` next to this notebook. Files stay on this machine.

In [ ]:
import io, zipfile, shutil, pathlib
SHOW_SOURCE = "show.zip"          # used when not on Colab: a folder of .txt files or a .zip
SHOW = pathlib.Path("show_input"); shutil.rmtree(SHOW, ignore_errors=True); SHOW.mkdir()
def _add(name, data):
    if name.lower().endswith(".zip"):
        with zipfile.ZipFile(io.BytesIO(data)) as z:
            for n in z.namelist():
                if n.lower().endswith(".txt") and not n.startswith("__MACOSX"):
                    (SHOW / pathlib.Path(n).name).write_bytes(z.read(n))
    elif name.lower().endswith(".txt"):
        (SHOW / pathlib.Path(name).name).write_bytes(data)
try:
    from google.colab import files
    for name, data in files.upload().items(): _add(name, data)
except ImportError:
    src = pathlib.Path(SHOW_SOURCE)
    for p in ([src] if src.is_file() else sorted(src.glob("*.txt"))): _add(p.name, p.read_bytes())
print(len(list(SHOW.glob("*.txt"))), "episode files")

## 2 · Parse (code, no AI)
Every non-camera line gets a stable ID (`E001`…). The LLM will only ever answer with these IDs, so it **cannot invent timestamps**.

In [ ]:
from promo_finder.parse import load_show, parse_report, compact_transcript
eps = load_show(SHOW)
ep_by_num = {e.number: e for e in eps}
report = parse_report(eps)
print({k: v for k, v in report.items() if not isinstance(v, (list, dict))})
print("\nEpisode 1, first lines the model sees:\n")
print("\n".join(compact_transcript(eps[0], seconds=True).splitlines()[:12]))

## 3 · Propose (the only AI judgement step)
One call per episode. The prompt asks for 0–5 promos as event-ID ranges with exact quotes and anchored 1–5 scores.

In [ ]:
from promo_finder.llm import LLMConfig
from promo_finder.prompts import EPISODE_SYSTEM, PROMPT_VERSION
GROQ_KEY = secret("GROQ_API_KEY", ask=True)
MODEL = "openai/gpt-oss-120b"   # or "openai/gpt-oss-20b", "qwen/qwen3.8-27b"; any OpenAI-compatible endpoint works via provider="custom"
cfg = LLMConfig.from_env("groq" if GROQ_KEY else "none", MODEL if GROQ_KEY else None, None, GROQ_KEY)
print("LLM:", cfg.describe() if cfg.enabled else "off -> rules-only baseline", "| prompt", PROMPT_VERSION)
print("\n" + EPISODE_SYSTEM[:1200], "…")

In [ ]:
import json, pathlib
from promo_finder.candidates import propose_episode, baseline_episode
EP = 1
if cfg.enabled:
    raw, status = propose_episode(ep_by_num[EP], eps, cfg, pathlib.Path("cache"))
else:
    raw, status = baseline_episode(ep_by_num[EP]), {"episode": EP, "note": "baseline"}
for i, p in enumerate(raw, 1): p["id"] = f"L{EP:03d}-{i}"
print(status)
print(json.dumps(raw[:2], indent=2, ensure_ascii=False)[:2500])

## 4 · Validate (code)
IDs → real screenplay times, snap to line edges, **hard rejects** (unknown IDs, outside 20–60s, outside the episode) and **soft flags** (slow open, opens on a pronoun, resolved ending, unexplained names, recap, quote mismatch).

In [ ]:
from promo_finder.validate import validate, known_names
names = known_names(eps)
checked = [validate(p, ep_by_num, names) for p in raw]
for c in checked:
    s = c["segments"][0] if c.get("segments") else {}
    print(f"{c['id']:9} {c['status']:9} {c.get('duration') or '':>6}s  {s.get('start_id','')}-{s.get('end_id','')}  "
          f"flags={c.get('flags')}  {c.get('reject_reason','')}")

## 5 · Rank (code)
`priority = 0.35·hook + 0.30·cliffhanger + 0.25·standalone + 0.10·funnel − flag penalty`. Each part blends the LLM's anchored score (70%) with a rule check (30%). Weights are product judgement, so the run also reports a ±0.1 sensitivity check.

In [ ]:
from promo_finder.rank import apply_scores, explain
valid = apply_scores([c for c in checked if c["status"] == "valid"], len(eps))
for c in sorted(valid, key=lambda c: -c["priority"]):
    print(f"{c['priority']:.2f}  {c['components']}  ->  {explain(c)}")

## 6 · Full show run
Same pipeline as the CLI and the Space. On the Groq free tier a full 50-episode show takes ~25 min (token-per-minute limits); set `EPISODES` for a quick demo. Re-runs hit the cache (0 new tokens).

In [ ]:
import pandas as pd
from promo_finder.pipeline import run_pipeline, to_rows
EPISODES = "1-10"      # None = whole show
res = run_pipeline(SHOW, cfg, out_root="outputs", episodes=EPISODES, cache_dir="cache",
                   progress=lambda i, n, ep: print(f"\r{i}/{n} episodes", end=""))
print("\n", res["meta"]["usage"], f"{res['meta']['elapsed_s']}s")
cols = ["rank", "segments", "duration_s", "priority", "hook", "cliffhanger", "post_reason", "flags"]
pd.DataFrame(to_rows(res["ranked"]))[cols].head(10)

In [ ]:
# The top promo as an editor sees it: real lines, HOOK and CLIFF marked
from html import escape as e
from IPython.display import HTML
from promo_finder.parse import fmt
eb = {x.number: x for x in res["episodes"]}
def excerpt(c):
    out, first, last = [], c["segments"][0], c["segments"][-1]
    for s in c["segments"]:
        out.append(f"<div style='color:#E25506;font-weight:700;margin-top:6px'>Episode {s['episode']} · {fmt(s['start'])} → {fmt(s['end'])}</div>")
        for ev in eb[s["episode"]].window(s["start"], s["end"]):
            tag = "HOOK" if (s is first and ev.id == first["start_id"]) else "CLIFF" if (s is last and ev.id == last["end_id"]) else ""
            bg = {"HOOK": "#fde2e3", "CLIFF": "#fff1d6"}.get(tag, "transparent")
            who = f"<b>{e(ev.speaker)}</b>: " if ev.kind == "DIALOGUE" else f"<i>{ev.kind.lower()}</i> · "
            out.append(f"<div style='background:{bg};padding:2px 6px'><code>{fmt(ev.start)}</code> <b>{tag}</b> {who}{e(ev.text)}</div>")
    return "<div style='font-family:sans-serif;font-size:13px'>" + "".join(out) + "</div>"
top = res["ranked"][0]
print(f"#1 priority {top['priority']:.2f} - {top['post_reason']}\nwhy: {top['why']}")
HTML(excerpt(top))

In [ ]:
# Honest quality checks from this run (auto metrics; the human blind review is in evaluate.py)
{k: v for k, v in res["metrics"].items() if k != "note"} | {"score_spread": res["meta"]["llm_score_spread"], "sensitivity": res["meta"]["weight_sensitivity"]}

## 7 · Visual prototype of the #1 promo (optional)
The LLM turns the real hook and cliffhanger lines into two shot prompts; **LTX-2.3** (free ZeroGPU Space) renders ~6s vertical clips with synced audio. If free GPU time is used up, the cell prints a storyboard instead. **Ranking never depends on this.**

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "gradio_client"], check=True)
from gradio_client import Client
from IPython.display import Video, display
from promo_finder.preview import shot_prompts, moments
P = shot_prompts(top, eb, cfg)
print("shot prompts by", P["source"], "\n\nHOOK SHOT:", P["hook_shot"], "\n\nCLIFF SHOT:", P["cliff_shot"])
try:
    ltx = Client("Lightricks/LTX-2-3", token=secret("HF_TOKEN"), verbose=False)
    for k in ("hook_shot", "cliff_shot"):
        path, _ = ltx.predict(input_image=None, prompt=P[k], duration=6, enhance_prompt=False, seed=10, randomize_seed=False,
                              height=896, width=512, api_name="/generate_video")      # sides must be multiples of 64
        if not path: raise RuntimeError("GPU Space returned no video")
        display(Video(path, embed=True, width=240))
except Exception as ex:
    head, tail, _ = moments(top, eb)
    print(f"\nVideo unavailable ({str(ex)[:120]}). Storyboard of the real cut:\n")
    print("[HOOK] ", *head[:3], sep="\n  ")
    print("  …")
    print("[CLIFF]", *tail[-3:], sep="\n  ")